# 03 — Shannon por partido (diversidad del bloque)

Shannon normalizado sobre el **perfil agregado** de cada partido: 0 = agenda focalizada en pocos temas, 1 = agenda repartida de forma amplia.


In [ ]:
# Cargar datos: cambia URL o XLSX segun corresponda
from pathlib import Path
import pandas as pd, numpy as np
from agendapp.io import fetch_endpoint, load_xlsx_municipio
from agendapp.transform import matriz_concejal_tema, binarizar, perfil_partido, filtrar_min_instrumentos
from agendapp.indices import shannon_norm, cv_shannon, jaccard_pairwise_mean, party_correlation
from agendapp import viz

# Opcion A: endpoint Apps Script (descomentar y poner URL real)
# URL = "https://script.google.com/macros/s/AKfycb.../exec"
# data = fetch_endpoint(URL)
# df_inst = pd.DataFrame(data["instrumentos"])

# Opcion B: xlsx local (piloto Guarne)
XLSX = Path("../..") / "Guarne_DILIGENCIADO.xlsx"
d = load_xlsx_municipio(XLSX)
df_inst = d["instrumentos"]
df_inst["Partido / Movimiento"] = df_inst["Partido / Movimiento"].astype(str).str.strip().str.upper()
df_inst.shape


In [ ]:
M = matriz_concejal_tema(df_inst, col_tema='Tematica', roles=('Proponente', 'Ponente', 'Coordinador'))
asign = (df_inst[df_inst['Rol'].isin(['Proponente', 'Ponente', 'Coordinador'])]
    .groupby('ID_Concejal')['Partido / Movimiento']
    .agg(lambda s: s.value_counts().idxmax()))
filas = []
for partido, cids in asign.groupby(asign).groups.items():
    sub = M.loc[[c for c in cids if c in M.index]]
    if sub.empty: continue
    perfil = perfil_partido(sub)
    filas.append({'partido': partido, 'n': sub.shape[0], 'shannon_partido': shannon_norm(perfil.values)})
h_df = pd.DataFrame(filas).set_index('partido')
h_df.sort_values('shannon_partido', ascending=False)


## Barras


In [ ]:
viz.barras_shannon_por_partido(h_df['shannon_partido'])
